In [ ]:
# Full Pipeline Execution - Direct Command Line
# Controls which replay mode results to display: "ideal", "realistic", or "dual" (side-by-side)
mode = "dual"

# Toggle this flag: True = interactive widget plots (slower), False = default static display (faster)
use_widget_plots = False

if use_widget_plots:
    %matplotlib widget

import concurrent.futures
from pathlib import Path
from IPython.display import display
from PIL import Image, ImageChops
import matplotlib.pyplot as plt

if use_widget_plots:
    plt.ion()
    plt.rcParams["figure.dpi"] = 110


def _trim_uniform_border(img, tolerance=8):
    rgb = img.convert("RGB")
    bg = Image.new("RGB", rgb.size, rgb.getpixel((0, 0)))
    diff = ImageChops.difference(rgb, bg)
    diff = diff.point(lambda p: 255 if p > tolerance else 0)
    bbox = diff.getbbox()
    return rgb.crop(bbox) if bbox else rgb


def show_image(img, title=None, width=10.5):
    if not use_widget_plots:
        display(img)
        return

    img = _trim_uniform_border(img)
    aspect = (img.width / img.height) if img.height else 1.0
    height = max(4, width / aspect)

    fig, ax = plt.subplots(figsize=(width, height))
    fig.canvas.toolbar_visible = True
    fig.canvas.header_visible = False
    fig.canvas.layout.width = "calc(100% - 56px)"
    fig.canvas.layout.margin = "0 0 0 56px"

    ax.imshow(img)
    ax.set_axis_off()
    ax.set_position([0.02, 0.01, 0.96, 0.95])

    if title:
        fig.suptitle(title, y=0.99)
    plt.show()


run_it = False
if run_it:
    import subprocess
    import concurrent.futures

    # Resolve project root
    project_root = Path.cwd().parent
    assert (project_root / "main_pipeline.py").exists(), f"main_pipeline.py not found at {project_root}"

    # Direct execution using virtual environment Python
    # Use the Python executable directly from the virtual environment
    venv_python = project_root / ".pythiaps" / "Scripts" / "python.exe"
    if venv_python.exists():
        # Use the virtual environment Python directly (fastest approach)
        result = subprocess.run(
            [str(venv_python), "main_pipeline.py", "--task", "all"], 
            cwd=str(project_root), 
            capture_output=True, 
            text=True
        )
    else:
        # Fallback: try direct python execution (may not work if venv not activated)
        result = subprocess.run(
            ["python", "main_pipeline.py", "--task", "all"], 
            cwd=str(project_root), 
            capture_output=True, 
            text=True
        )

    if result.returncode != 0:
        print(f"❌ Pipeline failed:")
        if result.stdout:
            print(result.stdout, end="")
        if result.stderr:
            print(result.stderr, end="")
        raise subprocess.CalledProcessError(result.returncode, result.args, output=result.stdout, stderr=result.stderr)

    # Print all output (both stdout and stderr)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")


def _find_plots_root(project_root, target_mode):
    """Return the most recent performance/plots dir for a specific resolved mode, or None."""
    candidates = sorted(
        project_root.glob(f"data/**/{target_mode}/performance/plots"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


def show_side_by_side(img_left, img_right, title_left, title_right, width=14.0):
    """Display two images side-by-side in a single matplotlib figure."""
    fig, axes = plt.subplots(1, 2, figsize=(width, width * img_left.height / (2 * img_left.width)))
    for ax, img, title in zip(axes, [img_left, img_right], [title_left, title_right]):
        ax.imshow(img)
        ax.set_title(title)
        ax.set_axis_off()
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot Grouping Overview
print("📊 Displaying Grouping Overview...")

if not run_it:
    from pathlib import Path
    from PIL import Image
    project_root = Path.cwd().parent

if mode == "dual":
    plots_ideal = _find_plots_root(project_root, "ideal")
    plots_realistic = _find_plots_root(project_root, "realistic")
    if plots_ideal is None or plots_realistic is None:
        print(f"Could not find plots for both modes. ideal={plots_ideal}, realistic={plots_realistic}")
    else:
        overview_ideal = plots_ideal / "gradient" / "gradient.png"
        overview_realistic = plots_realistic / "gradient" / "gradient.png"
        if overview_ideal.exists() and overview_realistic.exists():
            print("Showing grouping gradient (realistic | ideal)")
            show_side_by_side(
                Image.open(overview_realistic), Image.open(overview_ideal),
                "realistic", "ideal",
            )
        else:
            print("gradient.png missing for one or both modes.")
else:
    plots_root = _find_plots_root(project_root, mode)
    if plots_root is None:
        print(f"No grouping plots directory found for mode='{mode}'. Run the pipeline first.")
    else:
        overview = plots_root / "gradient" / "gradient.png"
        if overview.exists():
            print(f"Showing grouping gradient ({mode})")
            img = Image.open(overview)
            show_image(img, title=f"Grouping Gradient ({mode})", width=10.5)
        else:
            print("gradient/gradient.png not found.")
            print("Available files:")
            for p in sorted(plots_root.rglob("*.png")):
                print(" -", p.relative_to(plots_root))
            first_pngs = sorted(plots_root.rglob("*.png"))
            if first_pngs:
                print("Showing fallback:", first_pngs[0].name)
                img = Image.open(first_pngs[0])
                show_image(img, title=first_pngs[0].name, width=10.5)


In [ ]:
# Plot Perseries Performance (with parallel loading optimization)
def load_and_display_images(image_paths, max_images=12):
    """Load and display images in parallel for better performance"""
    def load_image(path):
        try:
            return path, Image.open(path)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return path, None

    # Load images in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        future_to_path = {executor.submit(load_image, path): path for path in image_paths[:max_images]}

        for future in concurrent.futures.as_completed(future_to_path):
            path, img = future.result()
            if img is not None:
                show_image(img, title=path.name, width=10.5)


def _show_perseries_for_root(plots_root, label):
    """Display trigger and signal perseries plots from a given plots root."""
    trigger_dir = plots_root / "trigger"
    if trigger_dir.exists():
        trigger_images = sorted(trigger_dir.glob("series_id*.png"))
        print(f"Found {len(trigger_images)} trigger perseries plots ({label}). Displaying first 12:")
        load_and_display_images(trigger_images, max_images=12)
    else:
        print(f"No trigger/ subfolder found ({label}).")

    signal_dir = plots_root / "signal"
    if signal_dir.exists():
        signal_images = sorted(signal_dir.glob("series_id*.png"))
        print(f"Found {len(signal_images)} signal perseries plots ({label}). Displaying first 12:")
        load_and_display_images(signal_images, max_images=12)
    else:
        print(f"No signal/ subfolder found ({label}).")


def _show_perseries_dual(plots_realistic, plots_ideal, subdir, max_images=12):
    """Show paired perseries plots side-by-side (realistic left, ideal right)."""
    dir_r = plots_realistic / subdir if plots_realistic else None
    dir_i = plots_ideal / subdir if plots_ideal else None
    has_r = dir_r is not None and dir_r.exists()
    has_i = dir_i is not None and dir_i.exists()
    if not has_r and not has_i:
        print(f"No {subdir}/ subfolder found for either mode.")
        return
    images_r = {p.name: p for p in sorted(dir_r.glob("series_id*.png"))} if has_r else {}
    images_i = {p.name: p for p in sorted(dir_i.glob("series_id*.png"))} if has_i else {}
    names = sorted(set(images_r) | set(images_i))[:max_images]
    print(f"Found {len(names)} {subdir} perseries plots (dual). Displaying first {min(len(names), max_images)}:")
    for name in names:
        if name in images_r and name in images_i:
            show_side_by_side(
                Image.open(images_r[name]), Image.open(images_i[name]),
                f"realistic – {name}", f"ideal – {name}",
            )
        elif name in images_r:
            show_image(Image.open(images_r[name]), title=f"realistic – {name}")
        else:
            show_image(Image.open(images_i[name]), title=f"ideal – {name}")


print("📊 Displaying Perseries Performance Plots...")

if mode == "dual":
    plots_ideal = _find_plots_root(project_root, "ideal")
    plots_realistic = _find_plots_root(project_root, "realistic")
    if plots_ideal is None and plots_realistic is None:
        print("No perseries plots found for either mode. Run the pipeline first.")
    else:
        _show_perseries_dual(plots_realistic, plots_ideal, "trigger")
        _show_perseries_dual(plots_realistic, plots_ideal, "signal")
else:
    plots_root = _find_plots_root(project_root, mode)
    if plots_root is None:
        print(f"No perseries plots found for mode='{mode}'. Run the pipeline first.")
    else:
        _show_perseries_for_root(plots_root, mode)
